# Tutorial 2 — Perturb-map held-out perturbation benchmark

The same held-out perturbation question as tutorial 1, on four sections of a spatial
Perturb-map (spatial Perturb-seq) assay. Running the benchmark on a second assay tests whether
the result depends on the measurement platform.

Read [tutorial 1](01_spac_seq_benchmark.ipynb) first — the scoring protocol, the two settings and
the two metrics are the same, and are explained there rather than repeated here.

### What differs from SPAC-seq

The model is the same one (`models/concert_benchmark`). The settings differ only where the two
assays differ:

| | SPAC-seq | Perturb-map |
|---|---|---|
| non-targeting control class | present | **absent** — unperturbed cells are the comparison group, `--control_label __NONE__` |
| guide assignment | ambiguous | unambiguous, so the technical correction is **off** (`use_action_correction=False`) |
| spot density | sparser, larger sections | denser, smaller sections |
| footprint inducing points | 16 | **64** |
| lengthscale band (normalized) | [0.15, 0.90] | **[0.01, 0.60]** |
| coordinate scale | 20 | **70** |
| minimum cells per perturbation | 25 | **0** — the panel is already balanced |

The inducing-point count and the training patience are not free choices: carrying over SPAC-seq's
16 inducing points and short patience costs about a third of the rank correlation on this assay.
64 points with patience 300 — the arm named **bandI** — is what the reported numbers come from,
and every one of its learned lengthscales sits inside the band rather than pinned at an end.

### Two extra traps specific to this assay

* **Scoring space.** CONCERT predicts its own 2000 highly-variable genes; the competitors score
  the 1053-gene assay panel. Scoring CONCERT over all 2000 measures spread over maximum-variance
  genes and inflates the error by an order of magnitude. The evaluation restricts to the
  panel overlap (~790 genes) so both sides are comparable.
* **Cell indexing.** The evaluation's prepared panel does not share barcodes or coordinates with
  the model's input files, so everything is scored in **model-row space** and the patch file is
  the bridge: patch cell → section pixel coordinate → array coordinate → model row.

## 0. Before you start

Run on a GPU node in `concert_env`; see tutorial 1, section 0, for the `salloc` recipe and the
GPU-compatibility note.

| Step | Time on one H100 |
|---|---|
| 2 — load checkpoint + one section | 1–3 min |
| 5 — one prediction | < 1 min |
| 6 — score one section | 5–15 min |
| 7, 8, 9 — reading shipped results | seconds |

## 1. Paths

`ms_pmap_niche_seen_bandI` is the checkpoint the paper reports. The other `band*` directories are
the sweep that chose it and are kept only for provenance — do not score them.

In [ ]:
import os, sys, json
from pathlib import Path
from types import SimpleNamespace

# --- code ---------------------------------------------------------------------------------
# The repository root: the directory holding models/ and benchmark_evaluation/. Found by
# walking up from the working directory, so the notebook runs from anywhere inside the repo;
# set CONCERT_CODEBASE to override. Every other root below reads an environment variable too,
# with the paths these results were produced on as the fallback -- see datasets/README.md.
def _repo_root():
    env = os.environ.get("CONCERT_CODEBASE")
    if env:
        return Path(env)
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "models" / "concert_benchmark").is_dir():
            return d
    raise RuntimeError("cannot locate the repository root; set CONCERT_CODEBASE")

def _root(var, default):
    return Path(os.environ.get(var, default))

CODEBASE    = _repo_root()
MODEL_SRC   = CODEBASE / "models" / "concert_benchmark"
EVAL_SRC    = CODEBASE / "benchmark_evaluation"
SCORING_LIB = _root("CONCERT_SCORING_LIB", "/n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/spac_seq_eval")
# Two metric helpers (eval_utils, spac_edist_align) are imported from an absolute path baked
# into spacseq_eval_common.EVAL_UTILS_DIR. It is not on sys.path here -- the module inserts it
# itself -- but it has to exist, so it is checked below.
HELPER_LIB  = _root("CONCERT_HELPER_LIB", "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/spaVAE/CONCERT-main/src/CONCERT-spac")

# --- data ---------------------------------------------------------------------------------
DATA      = _root("CONCERT_DATA_ROOT", "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/spaVAE/datasets")
PMAP      = _root("CONCERT_PMAP_EXP", "/n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/concert_data/exp/perturb_map")
PATCH_DIR = PMAP / "patches"                      # the niche patches, and the index bridge
PM_PREP   = PMAP / "prepared_panel1053ko"         # eval panel + prepared cell sets
HOLDOUT_COORDS = SCORING_LIB / "v3_holdout_coords"   # which cells were masked, per setting

# --- trained checkpoints ---------------------------------------------------------------------
OUTPUTS = _root("CONCERT_OUTPUT_ROOT", "/n/holylfs06/LABS/mzitnik_lab/Lab/xianglin226/CONCERT/outputs")
CKPT = {
    "seen":   OUTPUTS / "ms_pmap_niche_seen_bandI"   / "model.pt",   # the reported model
    "unseen": OUTPUTS / "ms_pmap_niche_unseen_bandI" / "model.pt",
}

# --- results that came with the paper --------------------------------------------------------
SHIPPED = {
    "seen":   PMAP / "results_panel1053ko_niche" / "concert_ms_seen_bandI",
    "unseen": PMAP / "results_panel1053ko_niche" / "concert_ms_unseen_bandI",
}
PAPER_NUMBERS = _root("CONCERT_PAPER_NUMBERS", "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/CONCERT/paper_package/numbers") / "benchmark_perturb_map"

WORKDIR = Path(os.environ.get(
    "CONCERT_NB_WORKDIR",
    "/n/holylabs/LABS/mzitnik_lab/Users/xianglin226/CONCERT/notebook_runs/02_perturb_map"))
WORKDIR.mkdir(parents=True, exist_ok=True)

SECTIONS = ["0", "1", "2", "3"]

checks = [
    ("model source",        MODEL_SRC / "concert_spac.py"),
    ("scoring code",        EVAL_SRC / "run_concert_pm_niche.py"),
    ("section s0",          DATA / "perturbmap_s0_spacfmt.h5ad"),
    ("niche patches",       PATCH_DIR / "slide0.npz"),
    ("eval panel",          PM_PREP / "benchmark_v3" / "eval_panel.json"),
    ("masked-cell coords",  HOLDOUT_COORDS / "holdout_v3_pmap_unseen.json"),
    ("seen checkpoint",     CKPT["seen"]),
    ("seen vocabulary",     CKPT["seen"].parent / "pert_map.json"),
    ("unseen checkpoint",   CKPT["unseen"]),
    ("shipped seen result", SHIPPED["seen"] / "seen_pert" / "slide_0" / "lfc_scores.json"),
    ("paper number table",  PAPER_NUMBERS / "pm_seen_lfc_dist_per_record.csv"),
]
for label, p in checks:
    print(f"{'OK ' if p.exists() else 'MISSING':>8}  {label:<22}  {p}")
print(f"\nnotebook output dir: {WORKDIR}")

## 2. Import the model and scoring code

Same import-order rule as tutorial 1: bind the released model copy into `sys.modules` before
anything else can put an older working tree in front of it.

In [ ]:
for p in (str(EVAL_SRC), str(MODEL_SRC)):
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(EVAL_SRC))
sys.path.insert(0, str(MODEL_SRC))

import numpy as np
import pandas as pd
import torch
import eval_spatial_counterfactual as ecf
import spacseq_eval_common as sec
import perturbmap_common as pm
import run_concert_pm_niche as pmeval        # the entry point the SLURM job runs

eu, sa = sec.import_eval_utils()

print("model source      :", ecf.__file__)
print("scoring entrypoint:", pmeval.__file__)
print("device            :", "cuda" if torch.cuda.is_available() else "CPU ONLY - too slow")
if torch.cuda.is_available():
    print("gpu               :", torch.cuda.get_device_name(0))

## 3. The evaluation configuration — bandI

The values below come from `experiments/02_perturb_seq_benchmark/train/train_perturb_seq.sh`
and its evaluation script. Read `fpert_ls_min` / `fpert_ls_max` **together with**
`fpert_coord_scale`: at scale 70 the band `[0.01, 0.60]` is `[0.7, 42]` in section units, a
section being 20 units wide. That is wide enough for the learned lengthscales to settle inside it.

As in tutorial 1, the band is a plain attribute of the field, not a saved weight — pass it at
evaluation or the same weights render a different field.

In [ ]:
def pmap_args(regime, outdir=None, slides=None, n_boot=10, v3_unseen=False):
    """Evaluation configuration for one Perturb-map run (bandI settings)."""
    return SimpleNamespace(
        model_file=CKPT[regime],
        outdir=Path(outdir) if outdir else (WORKDIR / f"{regime}_eval"),
        patch_dir=PATCH_DIR,
        slides=list(slides or SECTIONS),
        # --- scoring protocol ------------------------------------------------------------
        hvg_max_rank=2000,
        min_cells_per_pert=0,                  # the panel is balanced; no rare filter
        min_g_cells=20 if v3_unseen else 5,
        niche_grid=6, loc_range=20.0,
        n_samples=10, batch_size=512,
        max_src_cells=2000,
        n_boot=n_boot, boot_frac=0.5, boot_seed=1234,
        setup_only=False, fpert_off=False,
        # cross-slide: which cells were masked, and on which section
        v3_unseen_ko_file=(HOLDOUT_COORDS / "holdout_v3_pmap_unseen.json") if v3_unseen else None,
        # --- model graph: MUST match training --------------------------------------------
        use_action_correction=False,           # guide assignment is unambiguous here
        inducing_point_steps=4,
        niche_rank=4,
        use_expr_encoder=True, expr_hidden_dims=[512, 256], use_basal_encoder_only=True,
        use_nngp=False, nngp_lambda=0.0,
        use_paracrine=False, paracrine_l1=0.0,
        use_niche_interaction=False,           # FiLM head instead; auto-detected
        use_pert_gate=True, gate_hidden=64,
        spatial_basal_only=True,
        basal_field="rff", basal_bayes=True,
        # --- perturbation footprint field: the bandI settings -----------------------------
        fpert_field="gp_pp",
        fpert_rff_features=128, fpert_rff_bandwidth=0.15,
        fpert_coord_scale=70.0, fpert_hidden=256,
        fpert_svgp_inducing=64,                # self-corrected from the checkpoint anyway
        fpert_svgp_ls_init=0.04,
        fpert_mb_bandwidths="", f_pert_clamp=5.0,
        ls_prior_min=3.0, ls_prior_max=18.0,
        fpert_ls_min=0.01, fpert_ls_max=0.60,
    )

SECTION = "0"
args = pmap_args("seen", outdir=WORKDIR / "seen_eval", slides=[SECTION])
print(f"checkpoint  {args.model_file}")
print(f"band        [{args.fpert_ls_min}, {args.fpert_ls_max}] normalized "
      f"= [{args.fpert_ls_min * args.fpert_coord_scale:.1f}, "
      f"{args.fpert_ls_max * args.fpert_coord_scale:.1f}] section units "
      f"(a section is {args.loc_range:.0f} units wide)")

## 4. Load the checkpoint

`build_pm_ctx` does the same job as its SPAC-seq counterpart — read all four sections with the
shared gene mask, map labels to trained codes through `pert_map.json`, keep the target section,
build the graph, load the weights — and additionally reads the true inducing count and
niche-FiLM shape out of the checkpoint so a configuration mismatch cannot corrupt the field
silently.

In [ ]:
%%time
ctx = pmeval.build_pm_ctx(args, SECTION)

model = ctx["model"]
print()
print(f"section s{SECTION}: {ctx['N']:,} spots x {len(ctx['gene_names'])} genes")
print(f"perturbation vocabulary: {ctx['n_batch']} codes (code 0 = basal)")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print("perturbations:", sorted(k for k in ctx["name2code"] if ctx["name2code"][k] != 0))

## 5. Learned footprints, and one prediction

The per-perturbation footprint lengthscale is read out of the weights exactly as in tutorial 1 —
`ls(p) = ls_min + (ls_max - ls_min) * sigmoid(ls_head(e_p))` — but with this assay's band, so
the numbers come out in section units after multiplying by the coordinate scale.

This is the measurement behind the statement that the three knockouts have differently shaped
footprints: two of them diffuse, one focal.

In [ ]:
sd = torch.load(str(args.model_file), map_location="cpu", weights_only=False)
if isinstance(sd, dict) and "model_state_dict" in sd:
    sd = sd["model_state_dict"]

emb  = sd["lord_encoder.s_encoder.perturbation.weight"].float()
ls_w, ls_b = sd["fpert_field.ls_head.weight"].float(), sd["fpert_field.ls_head.bias"].float()
gs_w, gs_b = sd["g_scale_head.weight"].float(), sd["g_scale_head.bias"].float()

LS_MIN, LS_MAX, G_CAP, CS = args.fpert_ls_min, args.fpert_ls_max, 3.0, args.fpert_coord_scale
ls_norm = (LS_MIN + (LS_MAX - LS_MIN) * torch.sigmoid(emb @ ls_w.T + ls_b)).squeeze(1).numpy()
g_scale = torch.clamp(torch.nn.functional.softplus(emb @ gs_w.T + gs_b), max=G_CAP).squeeze(1).numpy()

code2name = ctx["pert_name"]
tbl = pd.DataFrame({
    "code": np.arange(len(ls_norm)),
    "perturbation": [code2name.get(i, f"code{i}") for i in range(len(ls_norm))],
    "lengthscale_norm": ls_norm,
    "lengthscale_section_units": ls_norm * CS,
    "niche_gain": g_scale,
}).iloc[1:]
print(f"band: [{LS_MIN * CS:.1f}, {LS_MAX * CS:.1f}] section units "
      f"-- none of these should be sitting on either end")
tbl.round(3)

In [ ]:
%%time
# The niche patches carry, per section, which cells are the held-out patch (role == 1) and
# which knockout dominates each niche. _model_adata maps all of that onto model rows.
# NOTE it rebuilds its own context, so it loads the checkpoint a second time (a minute or two).
adata_t, ctx2, gene_names, Yobs = pmeval._model_adata(args, SECTION)
groups = pm.seen_niche_groups(adata_t, src_role=1)
src_idx = pm.source_indices(adata_t, src_role=1)
panel = sorted({g for (_b, g, _r) in groups})
print(f"{len(groups)} niche groups, panel = {panel}, {len(src_idx)} source spots")

GENE = panel[0]
code_g = ctx["name2code"][GENE]
pred_g     = np.asarray(ecf._predict_swap(ctx, src_idx, code_g, args), dtype=np.float32)
pred_basal = np.asarray(ecf._predict_swap(ctx, src_idx, 0, args), dtype=np.float32)
model_lfc  = eu.pseudobulk_lfc(pred_g, pred_basal)

ko  = np.where(adata_t.obs["perturbation_canon"].astype(str).to_numpy() == GENE)[0]
ref = np.where(adata_t.obs["perturbation_canon"].astype(str).to_numpy() == pm.CTRL_CANON)[0]
obs_lfc = eu.pseudobulk_lfc(Yobs[ko], Yobs[ref])
print(f"measured: {len(ko)} {GENE} spots vs {len(ref)} control spots")

# restrict to the assay panel, as the scored comparison does
import h5py
with h5py.File(str(DATA / f"perturbmap_s{SECTION}_spacfmt.h5ad"), "r") as f:
    vnames = f["var"]["_index"][:].astype(str)
    inpanel = f["var"]["in_panel_1053"][:].astype(bool)
panel_set = set(vnames[inpanel])
mask = np.array([g in panel_set for g in gene_names], dtype=bool)
print(f"scoring {int(mask.sum())} of {len(gene_names)} genes (assay-panel overlap)")

scores = eu.score_lfc_agreement(model_lfc[mask], obs_lfc[mask])
print(f"\nsection-level illustration for {GENE} (NOT the reported per-niche score):")
for k, v in scores.items():
    print(f"  {k:<16} {v:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.scatter(obs_lfc[mask], model_lfc[mask], s=6, lw=0, alpha=0.4, color="#2166ac", rasterized=True)
lim = np.nanpercentile(np.abs(np.r_[obs_lfc[mask], model_lfc[mask]]), 99.5)
ax.plot([-lim, lim], [-lim, lim], lw=1, color="0.6", zorder=0)
ax.axhline(0, lw=0.6, color="0.85", zorder=0); ax.axvline(0, lw=0.6, color="0.85", zorder=0)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel(f"measured log fold change ({GENE})", fontsize=13)
ax.set_ylabel("predicted log fold change", fontsize=13)
ax.tick_params(labelsize=11)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
fig.tight_layout()
plt.show()

## 6. Score one section with the shipped harness

`run_seen` is what `evaluation/eval_perturb_seq.sh` calls. Per section it

1. maps the patch file onto model rows and rebuilds the observed matrix in log-CP10K,
2. finds the niches of the held-out patch and the knockout dominating each,
3. decodes every source spot to every panel perturbation, plus the basal slot,
4. restricts scoring to the assay-panel overlap and scores each (niche, knockout) record,
5. writes `lfc_scores.json` and `lfc_agg.json` under `seen_pert/slide_<n>/`.

Note it **skips** a section whose result file already exists — delete the directory to force a
recompute.

In [ ]:
%%time
RUN_SECTION_EVAL = True     # False -> read the shipped result instead

if RUN_SECTION_EVAL:
    pmeval.run_seen(args, SECTION)
    scores_path = args.outdir / "seen_pert" / f"slide_{SECTION}" / "lfc_agg.json"
else:
    scores_path = SHIPPED["seen"] / "seen_pert" / f"slide_{SECTION}" / "lfc_agg.json"

agg = json.loads(scores_path.read_text())["L2"]
print("read", scores_path, "\n")
pd.DataFrame([{"perturbation": g,
               "lfc_spearman": v["spearman_mean"],
               "lfc_rmse": v["lfc_dist_mean"],
               "n_niches": v["n_niches"],
               "n_scored_genes": v["n_scored_genes"]}
              for g, v in agg.items()]).sort_values("lfc_spearman", ascending=False)

## 7. The four sections together

One record is a (knockout, section, niche) triple, exactly as in tutorial 1. The cell reads the
shipped per-section results; point `RESULT_ROOT` at `args.outdir` once you have run step 6 for
all four sections.

In [ ]:
RESULT_ROOT = SHIPPED["seen"]      # or: WORKDIR / "seen_eval"

records = []
for s in SECTIONS:
    p = RESULT_ROOT / "seen_pert" / f"slide_{s}" / "lfc_scores.json"
    if not p.exists():
        print(f"  section {s}: no result at {p}")
        continue
    for gene, recs in json.loads(p.read_text())["L2"].items():
        for r in recs:
            records.append({"section": s, "perturbation": gene, "niche": r.get("niche"),
                            "lfc_spearman": r["spearman"], "lfc_rmse": r["lfc_dist"],
                            "n_scored_genes": r.get("n_scored_genes")})
pm_seen = pd.DataFrame(records)

print(f"{len(pm_seen)} records over {pm_seen.section.nunique()} sections")
print(f"\nWITHIN-NICHE (code name: seen)")
print(f"  LFC Spearman : {pm_seen.lfc_spearman.mean():.4f}")
print(f"  LFC RMSE     : {pm_seen.lfc_rmse.mean():.4f}")

pm_seen.groupby("perturbation").agg(n_records=("niche", "size"),
                                    lfc_spearman=("lfc_spearman", "mean"),
                                    lfc_rmse=("lfc_rmse", "mean")).round(4)

## 8. The cross-slide setting

Here a knockout is masked on its **target** section but kept on at least one donor section, so
the model has to place an effect it has only ever seen elsewhere. Which (knockout, section) pairs
are valid units is fixed by `benchmark_v3/eval_panel.json` — nothing else may be scored as
cross-slide.

The comparison is deliberately made identical to the competitors' harness: ground truth is the
masked knockout cells against the whole-section control population, with **no** niche
restriction (the record is registered with `niche = -1`). Scoring it against a niche-restricted
ground truth instead would make the numbers non-comparable.

**The cross-slide panel on this assay is tiny**: after the coverage requirements only one section
carries valid units, giving two records in total. Report it with that caveat, and do not read a
trend into it.

> **Retired numbers.** An older cross-slide variant pooled *donor* sections to build the ground
> truth instead of using the masked cells on the target section. Those numbers are invalid and
> were withdrawn. They still exist on disk in
> `paper_package/numbers/benchmark_perturb_map/perturbmap_unseen_per_pert.csv` (six records over
> four sections) — that file is CONCERT's own legacy table, **not** the benchmark, and must not be
> quoted as a cross-slide result.

In [ ]:
panel_json = json.loads((PM_PREP / "benchmark_v3" / "eval_panel.json").read_text())
print("cross-slide units (gene masked on target section, kept on donors):")
display(pd.DataFrame([{"gene": u["gene"], "target_section": u["target_slide"],
                       "donor_sections": ",".join(map(str, u.get("donor_slides", [])))}
                      for u in panel_json["eval_units"]]))

records = []
for s in SECTIONS:
    p = SHIPPED["unseen"] / "unseen_pert" / f"slide_{s}" / "lfc_scores.json"
    if not p.exists():
        continue
    for gene, recs in json.loads(p.read_text())["L2"].items():
        for r in recs:
            records.append({"section": s, "perturbation": gene,
                            "lfc_spearman": r["spearman"], "lfc_rmse": r["lfc_dist"]})
pm_unseen = pd.DataFrame(records)
print("\nCROSS-SLIDE (code name: unseen)")
print(f"  {len(pm_unseen)} records on {pm_unseen.section.nunique() if len(pm_unseen) else 0} section(s)"
      " -- too few to read a trend into")
print(f"  LFC Spearman : {pm_unseen.lfc_spearman.mean():.4f}")
print(f"  LFC RMSE     : {pm_unseen.lfc_rmse.mean():.4f}")
pm_unseen

To recompute the cross-slide results rather than read them, build the arguments with
`v3_unseen=True` and call the same `run_seen` — the flag switches the ground truth to the masked
cells and the output to `unseen_pert/`:

```python
u_args = pmap_args("unseen", outdir=WORKDIR / "unseen_eval", v3_unseen=True)
units_by_section = pmeval._v3_units_by_slide(u_args)
for s in SECTIONS:
    units = units_by_section.get(s, [])
    if units:                       # only target sections have units
        pmeval.run_seen(u_args, s, v3_units=units)
```

## 9. CONCERT next to the published methods

Same rule as tutorial 1: every method rescored under one recipe into a per-record CSV, and the
comparison **restricted to the records every method covers**.

This matters more here than on SPAC-seq. CONCERT's evaluation is missing four records the
competitors have (section 2 niche `3:*` for Ifngr2, section 3 `7:0` for Tgfbr2), so the
unrestricted counts are 22/3/2 against 23/6/2 — not like-for-like. The reported panel takes the
intersection itself; anything else reading this CSV has to do the same.

In this table the reported model is the method key **`concert`** (bandI). The keys
`concert_v3_seen` and `concert_niche_seen` are earlier checkpoints kept for provenance.
CellAgentChat cannot be scored on this assay at all — its output blocks are empty.

In [ ]:
df = pd.read_csv(PAPER_NUMBERS / "pm_seen_lfc_dist_per_record.csv")
df["record"] = df.gene.astype(str) + "|" + df.slide.astype(str) + "|" + df.niche_key.astype(str)

# the reported comparison: CONCERT (bandI) against the published methods only
KEEP = ["concert", "scgen", "biolord", "gears", "steamboat", "celcomen"]
sub = df[df.method.isin(KEEP)]
covered = sub.groupby("method").record.apply(set)
shared = set.intersection(*covered.tolist())
print(f"{len(shared)} of {sub.record.nunique()} records are covered by every method")
print("per method, before restriction:",
      {m: len(s) for m, s in covered.items()})

sub = sub[sub.record.isin(shared)]
overall = (sub.groupby(["method", "method_label"])
             .agg(n_records=("record", "size"),
                  lfc_spearman=("lfc_spearman", "mean"),
                  lfc_rmse=("lfc_dist", "mean"))
             .reset_index().sort_values("lfc_spearman", ascending=False))
display(overall.round(4))

per_gene = (sub.groupby(["method_label", "gene"])
              .agg(n=("record", "size"),
                   lfc_spearman=("lfc_spearman", "mean"),
                   lfc_rmse=("lfc_dist", "mean"))
              .reset_index())
per_gene.pivot(index="method_label", columns="gene", values="lfc_spearman").round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
d = overall.sort_values("lfc_spearman")
colors = ["#b2182b" if m == "concert" else "#9ab8d4" for m in d.method]
axes[0].barh(d.method_label, d.lfc_spearman, color=colors)
axes[0].axvline(0, lw=0.8, color="0.4")
axes[0].set_xlabel("LFC Spearman", fontsize=13)

d2 = overall.sort_values("lfc_rmse", ascending=False)
colors2 = ["#b2182b" if m == "concert" else "#9ab8d4" for m in d2.method]
axes[1].barh(d2.method_label, d2.lfc_rmse, color=colors2)
axes[1].set_xlabel("LFC RMSE (lower is better)", fontsize=13)

for ax in axes:
    ax.tick_params(labelsize=11)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
fig.tight_layout()
plt.show()

## What to watch out for

* **Score the assay panel, not all 2000 genes.** The evaluation restricts to the panel overlap.
  Scoring CONCERT's own highly-variable genes inflates the error roughly tenfold against methods
  that score the panel.
* **Restrict to shared records before comparing methods.** Four records are missing on CONCERT's
  side; without the intersection the comparison is not like-for-like.
* **Band, coordinate scale and inducing count must all match the checkpoint.** The inducing count
  is a saved tensor and would fail loudly, and the loader self-corrects it anyway; the band and
  the coordinate scale fail **silently**.
* **`ms_pmap_niche_seen_bandI` is the reported checkpoint.** The other `band*` directories are
  the sweep that selected it.
* **Cross-slide results here rest on two records.** State them with that caveat, and never quote
  the retired donor-pooled table (`perturbmap_unseen_per_pert.csv`) as a cross-slide result.
* **`seen` / `unseen` are directory names.** In text they are *within-niche prediction* and
  *cross-slide prediction*.

## Where things are

| | |
|---|---|
| model source | `codebase/models/concert_benchmark/` |
| scoring code | `codebase/benchmark_evaluation/` |
| experiment scripts | `codebase/experiments/02_perturb_seq_benchmark/` |
| reported tables | `paper_package/numbers/benchmark_perturb_map/` |